In [8]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

Global variables

In [2]:
DATASET_FOLDER = "../datasets/"
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 20

Load the dataset and replace the user and item ids so they are a contiguous sequence.

In [3]:
def LoadInteractions():
    train_interactions = pd.read_csv(f"{DATASET_FOLDER}/train_interactions.csv")

    train_interactions.rename(columns={'user_id': 'old_user_id', 'item_id': 'old_item_id'}, inplace=True)

    user_id_mapping = {val: i for i, val in enumerate(train_interactions['old_user_id'].unique())}
    train_interactions['user_id'] = train_interactions['old_user_id'].map(user_id_mapping)

    item_id_mapping = {val: i for i, val in enumerate(train_interactions['old_item_id'].unique())}
    train_interactions['item_id'] = train_interactions['old_item_id'].map(item_id_mapping)

    new_to_old_user_id_mapping = {v: k for k, v in user_id_mapping.items()}
    new_to_old_item_id_mapping = {v: k for k, v in item_id_mapping.items()}

    return train_interactions, new_to_old_user_id_mapping, new_to_old_item_id_mapping

Converts the interactions to a sparse interaction matrix

In [4]:
def CreateCSRMatrix(interactions):

    num_users = interactions['user_id'].unique().size
    num_items = interactions['item_id'].unique().size
    rows = interactions['user_id']
    cols = interactions['item_id']
    data = [1]*interactions['user_id'].size

    interaction_matrix_csr = csr_matrix((data, (rows, cols)), shape=(num_users, num_items))

    return interaction_matrix_csr

Two filter function to filter out users with low interactions and items with low interactions

In [5]:
def MinUsersPerItem(csr_matrix, item_mapping, min_users):
    # Filter out items with strictly less than min_users interactions
    item_mask = csr_matrix.getnnz(axis=0) >= min_users
    # Hint: use the .getnnz() method of the scipy csr matrix.

    # Apply the mask to the matrix to filter items
    filtered_interaction_matrix = csr_matrix[:, item_mask]

    # Great, because we're removing items (columns) from the matrix, we break our original mapping between columns and "old" item IDs!
    # Update the mapping:
    kept_cols = np.where(item_mask)[0]
    updated_item_mapping = {
        item_mapping[col]: new_idx
        for new_idx, col in enumerate(kept_cols)
    }
    
    return filtered_interaction_matrix, updated_item_mapping

def MinItemsPerUser(interaction_matrix_csr, user_mapping, min_items):
    # Filter out users with strictly less than min_items interactions
    user_mask = interaction_matrix_csr.getnnz(axis=1) >= min_items
    # Hint: use the .getnnz() method of the scipy csr matrix.
    
    # Apply the mask to the matrix to filter users
    filtered_interaction_matrix = interaction_matrix_csr[user_mask]

    # Great, because we're removing users (rows) from the matrix, we break our original mapping between rows and "old" user IDs!
    # Update the mapping:
    kept_rows = np.where(user_mask)[0]
    updated_user_mapping = {
        user_mapping[row]: new_idx
        for new_idx, row in enumerate(kept_rows)
    }
    
    return filtered_interaction_matrix, updated_user_mapping

def ApplyFilters(csr_matrix, user_mapping, item_mapping):
    filtered_matrix, updated_user_mapping = MinItemsPerUser(csr_matrix, user_mapping, MIN_USER_INTERACTIONS)
    filtered_matrix, updated_item_mapping = MinUsersPerItem(filtered_matrix, item_mapping, MIN_ITEM_INTERACTIONS)

    return filtered_matrix, updated_user_mapping, updated_item_mapping

In [6]:
def GetProcessedData():
    train_interactions, user_mapping, item_mapping = LoadInteractions()
    interaction_matrix_csr = CreateCSRMatrix(train_interactions)
    interaction_matrix_csr, user_mapping, item_mapping = ApplyFilters(interaction_matrix_csr, user_mapping, item_mapping)

    return interaction_matrix_csr, user_mapping, item_mapping

In [9]:
interaction_matrix_csr, user_mapping, item_mapping = GetProcessedData()

In [14]:

print("Now there are " + str(len(user_mapping)))

print("Now there are " + str(len(item_mapping)))

Now there are 46713
Now there are 4255
